In [1]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Python: e:\scFoundation_fork\model\.venv\Scripts\python.exe
PyTorch: 2.6.0+cu124
CUDA: True
GPU: NVIDIA RTX A6000


In [2]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import torch

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

REPO_ROOT = Path(r"E:\scFoundation_fork")
MODEL_DIR = REPO_ROOT / "model"

MODEL_PATH = MODEL_DIR / "models" / "models.ckpt"
INPUT_PATH = MODEL_DIR / "examples" / "smoke_test" / "test_cells.csv"
GENE_INDEX_PATH = MODEL_DIR / "OS_scRNA_gene_index.19264.tsv"

OUTPUT_DIR = REPO_ROOT / "outputs" / "scfoundation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# scFoundation source modules live here
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

# load.py expects some paths relative to model/
import os
os.chdir(MODEL_DIR)

from load import load_model_frommmf
from pretrainmodels import *


# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

SEED = 0

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ---------------------------------------------------------
# Device
# ---------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2,
        ),
        "GB",
    )


# ---------------------------------------------------------
# Load scFoundation gene vocabulary
# ---------------------------------------------------------

gene_list_df = pd.read_csv(
    GENE_INDEX_PATH,
    sep="\t",
)

gene_list = gene_list_df["gene_name"].tolist()

assert len(gene_list) == 19264

print("scFoundation genes:", len(gene_list))


# ---------------------------------------------------------
# Load expression input
# ---------------------------------------------------------

expression_df = pd.read_csv(
    INPUT_PATH,
    index_col=0,
)

print("Original input shape:", expression_df.shape)


# ---------------------------------------------------------
# Align input to scFoundation's 19,264 genes
# ---------------------------------------------------------

missing_genes = list(
    set(gene_list) - set(expression_df.columns)
)

padding_df = pd.DataFrame(
    np.zeros(
        (expression_df.shape[0], len(missing_genes))
    ),
    columns=missing_genes,
    index=expression_df.index,
)

expression_aligned = pd.concat(
    [expression_df, padding_df],
    axis=1,
)

expression_aligned = expression_aligned[gene_list]

assert expression_aligned.shape[1] == 19264

print("Aligned input shape:", expression_aligned.shape)
print("Missing genes zero-filled:", len(missing_genes))


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

TGTHIGHRES = "a5"
N_CONTEXTUAL_CELLS = 1

print()
print("Setup complete.")
print("Checkpoint:", MODEL_PATH)
print("Input:", INPUT_PATH)
print("Output:", OUTPUT_DIR)

Python: e:\scFoundation_fork\model\.venv\Scripts\python.exe
PyTorch: 2.6.0+cu124
Device: cuda
GPU: NVIDIA RTX A6000
Total VRAM: 44.99 GB
scFoundation genes: 19264
Original input shape: (5, 19263)
Aligned input shape: (5, 19264)
Missing genes zero-filled: 1

Setup complete.
Checkpoint: E:\scFoundation_fork\model\models\models.ckpt
Input: E:\scFoundation_fork\model\examples\smoke_test\test_cells.csv
Output: E:\scFoundation_fork\outputs\scfoundation


In [3]:
# test_cells.csv
#       ↓
# read gene symbols
#       ↓
# compare with scFoundation's 19,264-gene vocabulary
#       ↓
# zero-fill missing genes
#       ↓
# reorder columns exactly as expected by scFoundation
#       ↓
# expression_aligned

# That matches the repository's gene-selection behavior, which pads missing genes and reorders the matrix according to OS_scRNA_gene_index.19264.tsv.

In [4]:
# ============================================================
# CELL 1 — CELL EMBEDDING
# Output: N cells × 3072 dimensions
# ============================================================

from load import *


# ------------------------------------------------------------
# Load cell-embedding checkpoint
# ------------------------------------------------------------

model, config = load_model_frommmf(
    str(MODEL_PATH),
    "cell",
)

model = model.to(DEVICE)
model.eval()


# ------------------------------------------------------------
# Benchmark setup
# ------------------------------------------------------------

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start = time.perf_counter()

cell_embeddings = []


# ------------------------------------------------------------
# Process each cell
# ------------------------------------------------------------

with torch.no_grad():

    for i in range(expression_aligned.shape[0]):

        raw = expression_aligned.iloc[i].values.astype(np.float32)

        total_count = raw.sum()

        if total_count <= 0:
            raise ValueError(f"Cell {i} has zero total counts.")

        # Official scFoundation preprocessing for
        # single-cell input with pre_normalized=F
        normalized = np.log1p(
            raw / total_count * 1e4
        )

        # a5:
        # T = log10(total_count) + 5
        # S = log10(total_count)
        t_token = np.log10(total_count) + 5
        s_token = np.log10(total_count)

        model_input = np.concatenate(
            [
                normalized,
                [t_token, s_token],
            ]
        )

        model_input = torch.tensor(
            model_input,
            dtype=torch.float32,
            device=DEVICE,
        ).unsqueeze(0)

        gene_ids = torch.arange(
            19266,
            device=DEVICE,
        ).unsqueeze(0)

        value_labels = model_input > 0

        x, x_padding = gatherData(
            model_input,
            value_labels,
            config["pad_token_id"],
        )

        position_gene_ids, _ = gatherData(
            gene_ids,
            value_labels,
            config["pad_token_id"],
        )

        # Expression/value embedding
        x = model.token_emb(
            torch.unsqueeze(x, 2).float(),
            output_weight=0,
        )

        # Gene identity/position embedding
        position_emb = model.pos_emb(
            position_gene_ids
        )

        x = x + position_emb

        # Transformer encoder
        contextual = model.encoder(
            x,
            x_padding,
        )

        # Official pool_type="all"
        t_embedding = contextual[:, -1, :]
        s_embedding = contextual[:, -2, :]

        max_embedding, _ = torch.max(
            contextual[:, :-2, :],
            dim=1,
        )

        mean_embedding = torch.mean(
            contextual[:, :-2, :],
            dim=1,
        )

        # 768 × 4 = 3072
        cell_embedding = torch.cat(
            [
                t_embedding,
                s_embedding,
                max_embedding,
                mean_embedding,
            ],
            dim=1,
        )

        cell_embeddings.append(
            cell_embedding.cpu().numpy()
        )


# ------------------------------------------------------------
# Finish benchmark
# ------------------------------------------------------------

torch.cuda.synchronize()

elapsed = time.perf_counter() - start

peak_vram = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

cell_embeddings = np.concatenate(
    cell_embeddings,
    axis=0,
)


# ------------------------------------------------------------
# Validate and save
# ------------------------------------------------------------

output_path = OUTPUT_DIR / "cell_embeddings.npy"

np.save(
    output_path,
    cell_embeddings,
)

print()
print("scFoundation CELL EMBEDDING")
print("=" * 60)

print("Input cells:", expression_aligned.shape[0])
print("Input genes:", expression_aligned.shape[1])

print("Output shape:", cell_embeddings.shape)
print("Dimension:", cell_embeddings.shape[-1])
print("dtype:", cell_embeddings.dtype)

print("NaN:", np.isnan(cell_embeddings).sum())
print("Inf:", np.isinf(cell_embeddings).sum())

print("Time seconds:", round(elapsed, 4))
print(
    "Seconds per cell:",
    round(elapsed / expression_aligned.shape[0], 4),
)

print(
    "Peak GPU VRAM GB:",
    round(peak_vram, 2),
)

print("Saved:", output_path)

{'mask_gene_name': False, 'gene_num': 19266, 'seq_len': 19266, 'encoder': {'hidden_dim': 768, 'depth': 12, 'heads': 12, 'dim_head': 64, 'seq_len': 19266, 'module_type': 'transformer', 'norm_first': False}, 'decoder': {'hidden_dim': 512, 'depth': 6, 'heads': 8, 'dim_head': 64, 'module_type': 'performer', 'seq_len': 19266, 'norm_first': False}, 'n_class': 104, 'pad_token_id': 103, 'mask_token_id': 102, 'bin_num': 100, 'bin_alpha': 1.0, 'rawcount': True, 'model': 'mae_autobin', 'test_valid_train_idx_dict': '/nfs_beijing/minsheng/data/os10000w-new/global_shuffle/meta.csv.train_set_idx_dict.pt', 'valid_data_path': '/nfs_beijing/minsheng/data/valid_count_10w.npz', 'num_tokens': 13, 'train_data_path': None, 'isPanA': False, 'isPlanA1': False, 'max_files_to_load': 5, 'bin_type': 'auto_bin', 'value_mask_prob': 0.3, 'zero_mask_prob': 0.03, 'replace_prob': 0.8, 'random_token_prob': 0.1, 'mask_ignore_token_ids': [0], 'decoder_add_zero': True, 'mae_encoder_max_seq_len': 15000, 'isPlanA': False, 'ma

In [ ]:
# ============================================================
# CELL 2 — STATIC / BASE GENE IDENTITY EMBEDDING
# Source: pos_emb from scFoundation "gene" checkpoint
# Output: 19,264 genes × 768 dimensions
# ============================================================

from load import *


# ------------------------------------------------------------
# Load scFoundation cell model
# ------------------------------------------------------------

gene_model, gene_config = load_model_frommmf(
    str(MODEL_PATH),
    "gene",
)

gene_model = gene_model.to(DEVICE)
gene_model.eval()


# ------------------------------------------------------------
# Benchmark
# ------------------------------------------------------------

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start = time.perf_counter()

with torch.no_grad():

    gene_ids = torch.arange(
        19264,
        device=DEVICE,
    )

    # Fixed/base gene identity embeddings
    gene_embeddings = gene_model.pos_emb(
        gene_ids
    )


torch.cuda.synchronize()

elapsed = time.perf_counter() - start

peak_vram = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)


# ------------------------------------------------------------
# Move to CPU
# ------------------------------------------------------------

gene_embeddings = (
    gene_embeddings
    .float()
    .cpu()
    .numpy()
)


# ------------------------------------------------------------
# Validate and save
# ------------------------------------------------------------

output_path = OUTPUT_DIR / "gene_embeddings.npy"

np.save(
    output_path,
    gene_embeddings,
)

print()
print("scFoundation GENE EMBEDDING")
print("=" * 60)

print("Genes:", len(gene_list))
print("Output shape:", gene_embeddings.shape)
print("Dimension:", gene_embeddings.shape[-1])
print("dtype:", gene_embeddings.dtype)

print("NaN:", np.isnan(gene_embeddings).sum())
print("Inf:", np.isinf(gene_embeddings).sum())

print("Time seconds:", round(elapsed, 6))
print(
    "Peak GPU VRAM GB:",
    round(peak_vram, 2),
)

print("Saved:", output_path)

{'mask_gene_name': False, 'gene_num': 19266, 'seq_len': 19266, 'encoder': {'hidden_dim': 768, 'depth': 12, 'heads': 12, 'dim_head': 64, 'seq_len': 19266, 'module_type': 'transformer', 'norm_first': False}, 'decoder': {'hidden_dim': 512, 'depth': 6, 'heads': 8, 'dim_head': 64, 'module_type': 'performer', 'seq_len': 19266, 'norm_first': False}, 'n_class': 104, 'pad_token_id': 103, 'mask_token_id': 102, 'bin_num': 100, 'bin_alpha': 1.0, 'rawcount': True, 'model': 'mae_autobin', 'test_valid_train_idx_dict': '/nfs_beijing/minsheng/data/os10000w-new/global_shuffle/meta.csv.train_set_idx_dict.pt', 'valid_data_path': '/nfs_beijing/minsheng/data/valid_count_10w.npz', 'num_tokens': 13, 'train_data_path': None, 'isPanA': False, 'isPlanA1': False, 'max_files_to_load': 5, 'bin_type': 'auto_bin', 'value_mask_prob': 0.3, 'zero_mask_prob': 0.03, 'replace_prob': 0.8, 'random_token_prob': 0.1, 'mask_ignore_token_ids': [0], 'decoder_add_zero': True, 'mae_encoder_max_seq_len': 15000, 'isPlanA': False, 'ma

In [6]:
# ============================================================
# CELL 3 — CONTEXTUAL GENE EMBEDDING
# Output: N cells × 19,264 genes × 512 dimensions
# ============================================================

from load import *


# ------------------------------------------------------------
# Load gene-level scFoundation checkpoint
# ------------------------------------------------------------

context_model, context_config = load_model_frommmf(
    str(MODEL_PATH),
    "gene",
)

context_model = context_model.to(DEVICE)
context_model.eval()


# Required by official output_type="gene" path.
# Prevent final scalar/output projection so forward()
# returns the 3-D gene representation tensor.
context_model.to_final = None
# Limit by default because contextual output is large
n_cells = min(
    N_CONTEXTUAL_CELLS,
    expression_aligned.shape[0],
)

print("Contextual cells:", n_cells)


# ------------------------------------------------------------
# Benchmark
# ------------------------------------------------------------

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start = time.perf_counter()

contextual_embeddings = []


with torch.no_grad():

    for i in range(n_cells):

        raw = expression_aligned.iloc[i].values.astype(
            np.float32
        )

        total_count = raw.sum()

        if total_count <= 0:
            raise ValueError(
                f"Cell {i} has zero total counts."
            )

        # Official pre_normalized=F preprocessing
        normalized = np.log1p(
            raw / total_count * 1e4
        )

        # tgthighres = a5
        t_token = np.log10(total_count) + 5
        s_token = np.log10(total_count)

        pretrain_gene_x = np.concatenate(
            [
                normalized,
                [t_token, s_token],
            ]
        )

        pretrain_gene_x = torch.tensor(
            pretrain_gene_x,
            dtype=torch.float32,
            device=DEVICE,
        ).unsqueeze(0)

        (
            encoder_data,
            encoder_position_gene_ids,
            encoder_data_padding,
            encoder_labels,
            decoder_data,
            decoder_data_padding,
            new_data_raw,
            data_mask_labels,
            decoder_position_gene_ids,
        ) = getEncoerDecoderData(
            pretrain_gene_x,
            pretrain_gene_x,
            context_config,
        )

        out = context_model.forward(
            x=encoder_data,
            padding_label=encoder_data_padding,
            encoder_position_gene_ids=encoder_position_gene_ids,
            encoder_labels=encoder_labels,
            decoder_data=decoder_data,
            mask_gene_name=False,
            mask_labels=None,
            decoder_position_gene_ids=decoder_position_gene_ids,
            decoder_data_padding_labels=decoder_data_padding,
        )

        # Biological genes only
        out = out[:, :19264, :].contiguous()

        contextual_embeddings.append(
            out.float().cpu().numpy()
        )


torch.cuda.synchronize()

elapsed = time.perf_counter() - start

peak_vram = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)


# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

contextual_embeddings = np.concatenate(
    contextual_embeddings,
    axis=0,
)


# ------------------------------------------------------------
# Validate and save
# ------------------------------------------------------------

output_path = (
    OUTPUT_DIR / "contextual_gene_embeddings.npy"
)

np.save(
    output_path,
    contextual_embeddings,
)

print()
print("scFoundation CONTEXTUAL GENE EMBEDDING")
print("=" * 60)

print("Cells:", n_cells)
print("Genes/cell:", 19264)

print(
    "Output shape:",
    contextual_embeddings.shape,
)

print(
    "Dimension:",
    contextual_embeddings.shape[-1],
)

print(
    "dtype:",
    contextual_embeddings.dtype,
)

print(
    "NaN:",
    np.isnan(contextual_embeddings).sum(),
)

print(
    "Inf:",
    np.isinf(contextual_embeddings).sum(),
)

print(
    "Time seconds:",
    round(elapsed, 4),
)

print(
    "Seconds per cell:",
    round(elapsed / n_cells, 4),
)

print(
    "Peak GPU VRAM GB:",
    round(peak_vram, 2),
)

print(
    "Saved:",
    output_path,
)

{'mask_gene_name': False, 'gene_num': 19266, 'seq_len': 19266, 'encoder': {'hidden_dim': 768, 'depth': 12, 'heads': 12, 'dim_head': 64, 'seq_len': 19266, 'module_type': 'transformer', 'norm_first': False}, 'decoder': {'hidden_dim': 512, 'depth': 6, 'heads': 8, 'dim_head': 64, 'module_type': 'performer', 'seq_len': 19266, 'norm_first': False}, 'n_class': 104, 'pad_token_id': 103, 'mask_token_id': 102, 'bin_num': 100, 'bin_alpha': 1.0, 'rawcount': True, 'model': 'mae_autobin', 'test_valid_train_idx_dict': '/nfs_beijing/minsheng/data/os10000w-new/global_shuffle/meta.csv.train_set_idx_dict.pt', 'valid_data_path': '/nfs_beijing/minsheng/data/valid_count_10w.npz', 'num_tokens': 13, 'train_data_path': None, 'isPanA': False, 'isPlanA1': False, 'max_files_to_load': 5, 'bin_type': 'auto_bin', 'value_mask_prob': 0.3, 'zero_mask_prob': 0.03, 'replace_prob': 0.8, 'random_token_prob': 0.1, 'mask_ignore_token_ids': [0], 'decoder_add_zero': True, 'mae_encoder_max_seq_len': 15000, 'isPlanA': False, 'ma